# 03. Chemical Curation Visual Analysis

In this notebook, we visually inspect the outcome of `scripts/run_curation.py`:
1. Read the curated dataset from `data/processed/esol_curated.csv`.
2. View the audit table with side-by-side comparison of original vs. canonical SMILES.
3. Visually draw molecules before and after curation to verify structural fidelity.
4. Highlight why canonicalization is crucial for machine learning deduplication.

### 1. Load the Curated Dataset
Let's load the table generated by our pipeline in `data/processed/`.

In [ ]:
from pathlib import Path
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw

# Locate processed data robustly
data_path = Path("../data/processed/esol_curated.csv")
if not data_path.exists():
    data_path = Path("data/processed/esol_curated.csv")

df = pd.read_csv(data_path)
print(f"Total molecules loaded: {len(df)}")
print(f"Columns: {list(df.columns)}")
df[["Compound ID", "smiles", "curated_smiles", "measured log solubility in mols per litre"]].head()

### 2. Inspect Molecules Where SMILES Text Changed
Out of 1,128 molecules, exactly 624 (55.3%) changed their textual representation due to RDKit canonicalization.
Let's filter those 624 molecules and display the first 10 side-by-side in a clean comparison table:

In [ ]:
changed_mask = df["smiles"] != df["curated_smiles"]
changed_df = df[changed_mask][["Compound ID", "smiles", "curated_smiles"]].copy()

print(f"Total canonicalized molecules: {len(changed_df)} / {len(df)} ({len(changed_df)/len(df)*100:.1f}%)")
changed_df.head(10)

### 3. Visual Verification: Did the chemistry change or only the text?
Let's take 4 molecules whose SMILES changed significantly (e.g., Amygdalin, Fenfuram, Citral, Picene).
We parse both strings with RDKit and draw them side-by-side to verify that the **chemical structure is 100% identical**, only the atom indexing and canonical order changed.

In [ ]:
examples = changed_df.head(4)

mols_to_draw = []
legends = []

for _, row in examples.iterrows():
    name = row["Compound ID"]
    mol_orig = Chem.MolFromSmiles(row["smiles"])
    mol_curated = Chem.MolFromSmiles(row["curated_smiles"])
    
    mols_to_draw.extend([mol_orig, mol_curated])
    legends.extend([f"{name} (Orig)", f"{name} (Curated)"])

# Draw in a 2-column grid: Original on the left, Curated on the right
Draw.MolsToGridImage(mols_to_draw, legends=legends, molsPerRow=2, subImgSize=(350, 220))


### 4. Summary: Scientific Takeaway
- **Integrity:** Zero molecules were lost or broken.
- **Consistency:** 55.3% of the dataset now follows standard IUPAC/RDKit canonical atom numbering.
- **Ready for ML:** In the next week, when we generate Morgan Fingerprints and molecular descriptors, we will compute them on `curated_smiles` to guarantee perfect reproducibility.